# Interview domain analysis

This notebook calculates final domain-presence totals and independent-rating agreement for five interviews. It reads aggregate domain counts and independent rating-pair counts; individual interview profiles are not required. Domain definitions and methods are provided in the supplement.

Exact agreement includes matching uncertain ratings. Positive agreement uses definite present/absent pairs. Domain presence and coding agreement are reported separately.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data/processed"
OUTPUT = ROOT / "outputs"
OUTPUT.mkdir(parents=True, exist_ok=True)

domains = pd.read_csv(DATA / "interview_domains.csv")
totals = pd.read_csv(DATA / "interview_domain_totals.csv")
ratings = pd.read_csv(DATA / "interview_agreement_counts.csv")
STATES = {"present", "absent", "uncertain"}
assert len(domains) == 7 and domains.domain_id.is_unique
PRESENCE_COUNTS = ["present_interviews", "absent_interviews", "uncertain_interviews"]
assert len(totals) == 7 and totals.domain_id.is_unique
assert totals[PRESENCE_COUNTS].ge(0).all().all()
assert np.equal(totals[PRESENCE_COUNTS], np.floor(totals[PRESENCE_COUNTS])).all().all()
assert totals[PRESENCE_COUNTS].sum(axis=1).eq(5).all()
assert set(totals.domain_id) == set(domains.domain_id) == set(ratings.domain_id)
assert not ratings.duplicated(["domain_id", "assessment_round", "rating_a", "rating_b"]).any()
assert ratings.rating_a.isin(STATES).all() and ratings.rating_b.isin(STATES).all()
assert ratings.n.ge(0).all() and np.equal(ratings.n, np.floor(ratings.n)).all()
assert ratings.groupby("domain_id").n.sum().eq(5).all()
display(domains)
display(totals)


## Domain presence and agreement

In [ ]:
rows = []
for domain, group in ratings.groupby("domain_id", sort=True):
    n = int(group.n.sum())
    exact = int(group.loc[group.rating_a == group.rating_b, "n"].sum())
    definite = group.loc[(group.rating_a != "uncertain") & (group.rating_b != "uncertain")]
    positive = int(definite.loc[(definite.rating_a == "present") & (definite.rating_b == "present"), "n"].sum())
    discordant = int(definite.loc[definite.rating_a != definite.rating_b, "n"].sum())
    denominator = 2*positive+discordant
    presence = totals.set_index("domain_id").loc[domain]
    rows.append({"domain_id": domain, "assessment_round": group.assessment_round.iloc[0],
        "present_interviews": int(presence["present_interviews"]), "absent_interviews": int(presence["absent_interviews"]),
        "uncertain_interviews": int(presence["uncertain_interviews"]), "paired_interviews": n,
        "exact_agreement": exact/n, "positive_agreement": 2*positive/denominator if denominator else np.nan,
        "discrepancies": n-exact, "uncertain_pairs": n-int(definite.n.sum())})
summary = pd.DataFrame(rows).merge(domains[["domain_id", "domain_label"]], on="domain_id", validate="one_to_one")
display(summary)
summary.to_csv(OUTPUT / "interview_summary.csv", index=False)
